# ⚙️ Data Preprocessing & Feature Engineering
**Project ResistAI**

In this notebook, we transform our raw clinical data into a machine-learning-ready format. 

**Our Pipeline:**
1. **Reshape:** Convert "wide" format data to "long" format.
2. **Clean:** Remove missing values and anomalous species names (e.g., "?").
3. **Target Definition:** Binarize susceptibility into Resistant (1) vs. Susceptible (0).
4. **Feature Engineering:** Calculate Multi-Drug Resistance (MDR) count and Gram Stain.
5. **Encode:** Convert categorical text into numerical features using `LabelEncoder`.
6. **Balance:** Address class imbalances using SMOTE.

In [3]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore') # Keeps our notebook clean

In [4]:
# 1. Load the raw datasets
try:
    df1 = pd.read_csv("../data/raw/Bacteria_dataset_Multiresictance.csv")
    try:
        df2 = pd.read_excel("../data/raw/Dataset.xlsx")
    except FileNotFoundError:
        df2 = pd.read_csv("../data/raw/Dataset.csv")
        
    df_raw = pd.concat([df1, df2], ignore_index=True)
    print(f"Raw data loaded. Shape: {df_raw.shape}")
except Exception as e:
    print(f"Error loading data: {e}")

Raw data loaded. Shape: (10984, 33)


In [5]:
# 2. Reshape from Wide to Long format
abx_columns = [
    'AMX/AMP', 'AMC', 'CZ', 'FOX', 'CTX/CRO', 'IPM', 'GEN', 'AN', 
    'Acide nalidixique', 'ofx', 'CIP', 'C', 'Co-trimoxazole', 'Furanes', 
    'colistine', 'IMIPENEM', 'CEFTAZIDIME', 'GENTAMICIN', 'AUGMENTIN', 'CIPROFLOXACIN'
]

# Safely select only columns that actually exist
cols_to_melt = [col for col in abx_columns if col in df_raw.columns]

if 'Souches' in df_raw.columns:
    df = df_raw.melt(
        id_vars=['Souches'], 
        value_vars=cols_to_melt, 
        var_name='antibiotic', 
        value_name='susceptibility'
    )
    df = df.rename(columns={'Souches': 'species'})
    print(f"Reshaped to Long Format. New shape: {df.shape}")
    display(df.head(3))

Reshaped to Long Format. New shape: (219680, 3)


,species,antibiotic,susceptibility
0,S290 Escherichia coli,AMX/AMP,R
1,S291 Morganella morganii,AMX/AMP,S
2,S292 Proteus mirabilis,AMX/AMP,S


In [6]:
# 3. Clean Missing and Anomalous Data
df = df.dropna(subset=['susceptibility', 'species'])

# Remove the garbage species we found in EDA
bad_species = ['?', 'missing', 'Unknown']
df = df[~df['species'].isin(bad_species)]

# 4. Standardize Target Variable (Resistance)
res_mapping = {
    'Resistant': 1, 'R': 1, 'r': 1,
    'Susceptible': 0, 'S': 0, 's': 0,
    'Intermediate': 0, 'I': 0, 'i': 0 # We treat Intermediate as Susceptible for strict Resistance detection
}
df['resistance'] = df['susceptibility'].map(res_mapping)
df = df.dropna(subset=['resistance'])
df['resistance'] = df['resistance'].astype(int)

print(f"Cleaned dataset shape: {df.shape}")
print("\nResistance Distribution:")
print(df['resistance'].value_counts(normalize=True) * 100)

Cleaned dataset shape: (149205, 4)

Resistance Distribution:
resistance
0    67.895848
1    32.104152
Name: proportion, dtype: float64


In [7]:
# 5. Feature Engineering
print("Engineering biological features...")

# --- A. Multi-Drug Resistance (MDR) Count ---
# Since we don't have distinct isolate IDs in this merged dataset, 
# we'll use a heuristic MDR count based on the species' overall resistance average.
species_mdr = df.groupby('species')['resistance'].sum().to_dict()
df['multi_drug_count'] = df['species'].map(species_mdr)

# Normalize the MDR count to a 0-15 scale to prevent massive outliers
max_mdr = df['multi_drug_count'].max()
if max_mdr > 0:
    df['multi_drug_count'] = (df['multi_drug_count'] / max_mdr) * 15
df['multi_drug_count'] = df['multi_drug_count'].astype(int)

# --- B. Gram Stain Heuristic ---
gram_neg_genera = [
    'Escherichia', 'Klebsiella', 'Pseudomonas', 'Acinetobacter', 
    'Salmonella', 'Enterobacter', 'Proteus', 'Serratia', 'Citrobacter'
]

def determine_gram(species_name):
    species_str = str(species_name)
    if any(genus in species_str for genus in gram_neg_genera):
        return 1 # Negative
    return 0     # Positive

df['gram_stain_enc'] = df['species'].apply(determine_gram)

# --- C. Handle Missing UI Inputs ---
# Streamlit expects 'mic_value' and 'isolation_source'. Let's add standard defaults.
df['mic_value'] = 1.0 
df['isolation_source'] = 'Clinical Sample'

display(df[['species', 'multi_drug_count', 'gram_stain_enc']].drop_duplicates().head(5))

Engineering biological features...


,species,multi_drug_count,gram_stain_enc
0,S290 Escherichia coli,3,1
1,S291 Morganella morganii,1,0
2,S292 Proteus mirabilis,2,1
4,S294 Escherichia coli,3,1
5,S295 Escherichia coli,2,1


In [8]:
# 6. Encode Categorical Variables
encoders = {}
cols_to_encode = ['species', 'antibiotic', 'isolation_source']

for col in cols_to_encode:
    df[col] = df[col].astype(str).fillna('Unknown')
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    encoders[col] = le

# Ensure models directory exists to save encoders
os.makedirs("../models", exist_ok=True)
joblib.dump(encoders, "../models/encoders.pkl")
print("✅ Label Encoders saved to models/encoders.pkl")

✅ Label Encoders saved to models/encoders.pkl


In [9]:
# 7. Balance Data using SMOTE
feature_cols = ['species_enc', 'antibiotic_enc', 'isolation_source_enc', 'mic_value', 'multi_drug_count', 'gram_stain_enc']

X = df[feature_cols]
y = df['resistance']

print(f"Pre-SMOTE shape: {X.shape}")

# Only apply SMOTE if there is a severe imbalance (e.g., one class is < 40%)
if y.mean() < 0.40 or y.mean() > 0.60:
    print("Class imbalance detected. Applying SMOTE...")
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)
else:
    print("Classes are relatively balanced. Skipping SMOTE.")
    X_resampled, y_resampled = X, y

print(f"Post-SMOTE shape: {X_resampled.shape}")

# Recombine and save to processed directory
df_final = X_resampled.copy()
df_final['resistance'] = y_resampled

# Add the text columns back in for the Streamlit UI Heatmap to use!
# (We inverse_transform the encoded columns)
for col in cols_to_encode:
    df_final[col] = encoders[col].inverse_transform(df_final[col + '_enc'])

os.makedirs("../data/processed", exist_ok=True)
df_final.to_csv("../data/processed/final_dataset.csv", index=False)
print("🎉 Final machine-learning-ready dataset saved to data/processed/final_dataset.csv!")

Pre-SMOTE shape: (149205, 6)
Class imbalance detected. Applying SMOTE...
Post-SMOTE shape: (202608, 6)
🎉 Final machine-learning-ready dataset saved to data/processed/final_dataset.csv!
